# Validação T4 (sm_75) — Dory 4B FP16 & INT8 (GPTQ W8A16)

Este notebook executa **de ponta a ponta** a validação do `nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-BF16` em uma **T4 (sm_75)** antes de gerar a F1a nas RTX 2060 SUPER do laboratório. Toda a lógica fica nos scripts (`scripts/t4_validate.py` e `scripts/patch_vllm_dory.py`); cada passo grava no Drive e **pula o que já estiver pronto**. Se uma célula falhar, a execução **para imediatamente** para não gastar créditos à toa.

## Roteiro & Custos Esperados (T4 ≈ 2 CU/h)
| Célula | Passo | Duração estimada | Custo aprox. (CU) |
|---|---|---|---|
| **0** | Drive + `HF_TOKEN` + `git clone/pull` + trava GPU T4 (`sm_75`) | < 1 min | ~0,02 |
| **1** | `colab_install.sh` (wheels no Drive) + `patch_vllm_dory.py` + `--check` | 15–25 min na 1ª vez; ~1–2 min depois | ~0,5–0,8 (1ª vez) / ~0,05 |
| **2** | `t4_validate.py env` (PyTorch, vLLM, VRAM, matmul BF16) | < 1 min | ~0,02 |
| **3** | `t4_validate.py quantize` (RTN INT8 W8A16 g128, sem `g_idx`, `dtype=float16`) | ~2–4 min | ~0,1 |
| **4** | `dump` (`int8` L2 primeiro + retry `FLEX_ATTENTION` se falhar; `fp16` L2; L1/L4 depois) | ~5–10 min | ~0,2–0,3 |
| **5** | `t4_validate.py compare` (HF vs vLLM; fallback para `cpu-bf16` se `gpu-bf16` falhar) | ~3–8 min | ~0,1–0,25 |
| **6** | `smoke --variant int8` (KV + tok/s + estimativa de CU) + `quality` opcional (`RUN_QUALITY=False`) | ~2 min (`smoke`); `quality` conforme estimativa | ~0,07 (`smoke`) |
| **7** | `t4_validate.py report` (tabela final + `report.json`) | < 1 min | ~0,01 |

## O que fazer se cada passo falhar
- **Célula 0 (GPU não é T4 / sm_75):** troque o tipo de runtime no Colab para **T4 GPU** antes de continuar. Não gaste CU em L4/A100/CPU aqui.
- **Célula 1 (`colab_install.sh` ou `patch_vllm_dory.py`):** se `patch_vllm_dory.py` reclamar de âncora ausente, a wheel do Drive veio de outro commit do vLLM — apague a pasta `$WHEELS` no Drive e rode novamente.
- **Célula 3 (`quantize`):** se faltar disco local, limpe `/content` ou caches antigos; se faltar espaço no Drive, o script pula a cópia ao Drive com aviso (o checkpoint local em `/content/dory_int8_gptq` continua válido nesta sessão).
- **Célula 4 (`dump`):** se `int8 L2` falhar no backend padrão e também com `--attn-backend FLEX_ATTENTION`, **pare** e inspecione as linhas de erro/traceback impressas (não rode L1/L4 nem `fp16`).
- **Célula 5 (`compare`):** se `gpu-bf16` falhar no SDPA da T4, o script cai para `cpu-bf16`; em Colab padrão (~12,7 GB RAM) pode dar OOM — mude para **High-RAM** (os resultados anteriores já estão salvos no Drive).
- **Célula 6 (`smoke` / `quality`):** confira `max_concurrency` (ideal >= 2 para `max_model_len=12288`) e `est_cu` antes de mudar `RUN_QUALITY = True`. Se `max_concurrency < 2`, reduza ou ajuste `LAB_VRAM_GIB` / `--batch 1`.


In [ ]:
# Célula 0 — Drive, HF_TOKEN opcional, repositório, trava de GPU T4 (sm_75) e helper de log
import os
import subprocess
from pathlib import Path
import torch
from google.colab import drive, userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # opcional, acelera downloads
except Exception:
    print("HF_TOKEN não configurado (ok, só fica mais lento).")

os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/root/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/root/.cache/huggingface/datasets"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

drive.mount("/content/drive")

os.environ.setdefault("CODER_DRIVE_ROOT", "/content/drive/MyDrive/coder-comprimido")  # raiz do Drive do projeto (troque para o atalho do Drive central)
DRIVE_ROOT = Path(os.environ["CODER_DRIVE_ROOT"])
OUT = DRIVE_ROOT / "results" / "t4_validation"
INT8_LOCAL = Path("/content/dory_int8_gptq")
INT8_DRIVE = DRIVE_ROOT / "checkpoints" / "dory_int8_gptq_rtn_g128"
OUT.mkdir(parents=True, exist_ok=True)

REPO = "/content/experimentos-llm-local"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/Arthur-galvao/experimentos-llm-local.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
%cd {REPO}

if not torch.cuda.is_available():
    raise RuntimeError("Nenhuma GPU CUDA disponível. Selecione runtime T4 GPU no Colab.")
props = torch.cuda.get_device_properties(0)
cap = f"{props.major}.{props.minor}"
print(f"GPU detectada: {props.name} | sm_{props.major}{props.minor} | VRAM={props.total_memory / 2**30:.2f} GiB")
if cap != "7.5" or "T4" not in props.name.upper():
    raise RuntimeError(f"GPU {props.name} (sm_{props.major}{props.minor}) não é T4/sm_75. Parando para não gastar CU.")


def run_step(cmd, *, raise_on_error=True):
    """Executa subprocesso capturando stdout+stderr e imprimindo causa/traceback em caso de erro."""
    print(">>>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    linhas = (r.stdout + "\n" + r.stderr).strip().splitlines()
    if r.returncode != 0:
        erros = [l for l in linhas if "Error" in l or "ERROR" in l or "error:" in l]
        if erros:
            print("--- linhas de erro ---")
            print("\n".join(erros[-25:]))
        idx = next((i for i, l in enumerate(linhas) if "Traceback" in l), max(0, len(linhas) - 80))
        print("--- início do primeiro traceback ---")
        print("\n".join(linhas[max(0, idx - 5) : idx + 40]))
        print("--- final do log ---")
    print("\n".join(linhas[-40:]))
    if raise_on_error and r.returncode != 0:
        raise RuntimeError(f"Comando falhou (código {r.returncode}): {' '.join(cmd)}")
    return r


In [ ]:
# Célula 1 — Instalação de dependências (wheels do Drive) + patch do vLLM Dory + --check
if "run_step" not in globals():
    raise RuntimeError("Execute a Célula 0 antes.")

chk = run_step(["python", "scripts/patch_vllm_dory.py", "--check"], raise_on_error=False)
if chk.returncode == 0:
    print("[SKIP] vLLM + patch do Dory já estão instalados e verificados.")
else:
    run_step(["bash", "scripts/colab_install.sh"])
    run_step(["python", "scripts/patch_vllm_dory.py"])
    run_step(["python", "scripts/patch_vllm_dory.py", "--check"])


In [ ]:
# Célula 2 — Diagnóstico do ambiente (`t4_validate.py env`)
run_step(["python", "scripts/patch_vllm_dory.py", "--check"])

env_path = OUT / "env.json"
if env_path.exists():
    print(f"[SKIP] {env_path} já existe:")
    print(env_path.read_text(encoding="utf-8"))
else:
    run_step(["python", "scripts/t4_validate.py", "env"])


In [ ]:
# Célula 3 — Quantização RTN INT8 W8A16 (`t4_validate.py quantize`)
if not (OUT / "env.json").exists():
    raise RuntimeError("Célula 2 (`env`) não concluída: env.json ausente.")

if (INT8_DRIVE / "config.json").exists() or (INT8_LOCAL / "config.json").exists():
    onde = INT8_LOCAL if (INT8_LOCAL / "config.json").exists() else INT8_DRIVE
    print(f"[SKIP] Checkpoint INT8 já existe em {onde}.")
else:
    run_step(["python", "scripts/t4_validate.py", "quantize"])
    if not (INT8_LOCAL / "config.json").exists() and not (INT8_DRIVE / "config.json").exists():
        raise RuntimeError("quantize terminou sem gravar config.json.")


In [ ]:
# Célula 4 — Dumps vLLM (`int8` L2 primeiro com retry FLEX_ATTENTION; depois `fp16` L2; L1/L4 se L2 passou)
run_step(["python", "scripts/patch_vllm_dory.py", "--check"])
if not ((INT8_LOCAL / "config.json").exists() or (INT8_DRIVE / "config.json").exists()):
    raise RuntimeError("Célula 3 (`quantize`) não concluída: checkpoint INT8 ausente.")

INT8_ATTN_BACKEND = None


def _dump_one(variant, loops, attn_backend=None, *, raise_on_error=True):
    out_file = OUT / "dumps" / f"{variant}_L{loops}.json"
    if out_file.exists():
        print(f"[SKIP] {out_file} já existe.")
        return 0
    cmd = ["python", "scripts/t4_validate.py", "dump", "--variant", variant, "--loops", str(loops)]
    if attn_backend:
        cmd += ["--attn-backend", attn_backend]
    res = run_step(cmd, raise_on_error=raise_on_error)
    return res.returncode


# 1) INT8 L2 primeiro; se falhar no backend padrão, tenta UMA vez com FLEX_ATTENTION
if (OUT / "dumps" / "int8_L2.json").exists():
    print("[SKIP] int8_L2.json já existe.")
else:
    rc = _dump_one("int8", 2, raise_on_error=False)
    if rc != 0:
        print("\n[RETRY] int8 L2 falhou no backend padrão; tentando UMA vez com --attn-backend FLEX_ATTENTION...")
        _dump_one("int8", 2, attn_backend="FLEX_ATTENTION", raise_on_error=True)
        INT8_ATTN_BACKEND = "FLEX_ATTENTION"

if not (OUT / "dumps" / "int8_L2.json").exists():
    raise RuntimeError("int8_L2.json não foi gerado. Parando antes do fp16/L1/L4.")

# 2) FP16 L2
_dump_one("fp16", 2, raise_on_error=True)
if not (OUT / "dumps" / "fp16_L2.json").exists():
    raise RuntimeError("fp16_L2.json não foi gerado. Parando antes de L1/L4.")

# 3) L1 e L4 só rodam se ambos L2 passaram
for k in (1, 4):
    _dump_one("int8", k, attn_backend=INT8_ATTN_BACKEND, raise_on_error=True)
for k in (1, 4):
    _dump_one("fp16", k, raise_on_error=True)


In [ ]:
# Célula 5 — Comparação teacher-forced HF vs vLLM (`t4_validate.py compare`)
dumps = sorted((OUT / "dumps").glob("*.json"))
if not (OUT / "dumps" / "int8_L2.json").exists() or not (OUT / "dumps" / "fp16_L2.json").exists():
    raise RuntimeError("Célula 4 (`dump`) incompleta: int8_L2.json e fp16_L2.json são obrigatórios.")

pendentes = [d for d in dumps if not (OUT / f"equiv_{d.stem}.json").exists()]
if not pendentes:
    print(f"[SKIP] Todos os {len(dumps)} dumps já possuem equiv_*.json no Drive.")
else:
    run_step(["python", "scripts/t4_validate.py", "compare"])


In [ ]:
# Célula 6 — Smoke 12288 (`smoke --variant int8`) + estimativa de CU + `quality` opcional
import json

run_step(["python", "scripts/patch_vllm_dory.py", "--check"])
if not (OUT / "equiv_int8_L2.json").exists():
    raise RuntimeError("Célula 5 (`compare`) não concluída: equiv_int8_L2.json ausente.")

RUN_QUALITY = False  # mude para True somente após conferir a estimativa de CU abaixo
N_PROBLEMS = 32

smoke_path = OUT / "smoke_int8.json"
if smoke_path.exists():
    print(f"[SKIP] {smoke_path} já existe.")
else:
    cmd_smoke = ["python", "scripts/t4_validate.py", "smoke", "--variant", "int8", "--n-problems", str(N_PROBLEMS)]
    if globals().get("INT8_ATTN_BACKEND"):
        cmd_smoke += ["--attn-backend", INT8_ATTN_BACKEND]
    run_step(cmd_smoke)

smoke = json.loads(smoke_path.read_text(encoding="utf-8"))
print(
    f"[SMOKE INT8] tok/s={smoke['tok_s']} | max_concurrency={smoke.get('max_concurrency')} | "
    f"estimativa quality ({N_PROBLEMS} probs): {smoke['est_hours']} h ≈ {smoke['est_cu']} CU"
)

if RUN_QUALITY:
    cmd_q = ["python", "scripts/t4_validate.py", "quality", "--variant", "int8", "--n-problems", str(N_PROBLEMS)]
    if globals().get("INT8_ATTN_BACKEND"):
        cmd_q += ["--attn-backend", INT8_ATTN_BACKEND]
    run_step(cmd_q)
else:
    print("RUN_QUALITY=False — altere para True acima se aprovar a estimativa de CU e quiser rodar o quality.")


In [ ]:
# Célula 7 — Relatório final consolidado (`t4_validate.py report`)
if not (OUT / "smoke_int8.json").exists() and not list(OUT.glob("equiv_*.json")):
    raise RuntimeError("Nenhum resultado de compare ou smoke encontrado para gerar o relatório.")

run_step(["python", "scripts/t4_validate.py", "report"])
